In [3]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from matplotlib.image import imread
from PIL import Image
import tensorflow as tf
np.random.seed(1337)
import gc

from tensorflow.keras import layers
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Activation, Dropout, AveragePooling2D,Flatten, Dense, Conv2D,MaxPool2D, MaxPooling2D, BatchNormalization,Conv2DTranspose,concatenate
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')


img_size = 128
print(os.listdir())
dataset = os.listdir("music_seperation_dataset/train")
labels = dataset
print(labels)

['.git', '.vscode', 'model_classes_3.ipynb', 'model_classes_full.ipynb', 'model_separation.ipynb', 'music_dataset_spectro_3_instrument', 'music_dataset_spectro_full', 'music_seperation_dataset', 'my_model_3.keras', 'my_model_full.keras', 'README.md', 'spectrogramMaker.py']
['Acoustic_Guitar', 'Bass_Guitar', 'Drum_set', 'Electric_Guitar', 'full_mix', 'Keyboard']


In [2]:
def get_dataset_array(data_dir):
    data = []
    for label in labels:
        print(label)
        path = os.path.join(data_dir, label)
        class_num = labels.index(label)
        for img in os.listdir(path):
            try:
                img_arr = cv2.imread(os.path.join(path, img))
                resized_arr = cv2.resize(img_arr, (img_size, img_size))
                data.append([resized_arr, class_num])
                gc.collect()
            except Exception as e:
                print(e)
    return np.array(data,dtype="object")

In [ ]:
train = get_dataset_array("music_seperation_dataset/train/")
test = get_dataset_array("music_seperation_dataset/test/")
valid = get_dataset_array("music_seperation_dataset/valid/")

Accordion
Acoustic_Guitar
Banjo
Bass_Guitar
Clarinet
cowbell
Dobro
Drum_set
Electric_Guitar
flute
Harmonium
Horn
Keyboard
Mandolin
Organ
Piano
Saxophone
Shakers
Tambourine
Trombone
Trumpet
Ukulele
vibraphone
Violin
Accordion
Acoustic_Guitar
Banjo
Bass_Guitar
Clarinet
cowbell
Dobro
Drum_set
Electric_Guitar
flute
Harmonium
Horn
Keyboard
Mandolin
Organ
Piano
Saxophone
Shakers
Tambourine
Trombone
Trumpet
Ukulele
vibraphone
Violin
Accordion
Acoustic_Guitar
Banjo
Bass_Guitar
Clarinet
cowbell
Dobro
Drum_set
Electric_Guitar
flute
Harmonium
Horn
Keyboard
Mandolin
Organ
Piano
Saxophone
Shakers
Tambourine
Trombone
Trumpet
Ukulele
vibraphone
Violin


In [4]:
print(train.shape)
print(test.shape)
print(valid.shape)

(32888, 2)
(4111, 2)
(4123, 2)


In [5]:
x_train = []
y_train = []

x_val = []
y_val = []

x_test = []
y_test = []

for feature, label in train:
    x_train.append(feature)
    y_train.append(label)

for feature, label in test:
    x_test.append(feature)
    y_test.append(label)

for feature, label in valid:
    x_val.append(feature)
    y_val.append(label)
     
del train
del test
del valid

In [6]:
gc.collect()
x_train = np.array(x_train)/255
gc.collect()
x_test = np.array(x_test)/255
gc.collect()
x_val = np.array(x_val)/255
gc.collect()

0

In [7]:
x_train = x_train.reshape(-1, img_size, img_size, 3)
y_train = np.array(y_train)

x_val = x_val.reshape(-1, img_size, img_size, 3)
y_val = np.array(y_val)

x_test = x_test.reshape(-1, img_size, img_size, 3)
y_test = np.array(y_test)


In [5]:

def Unet():
    inputs =  layers.Input(shape=(128,128,3))

    conv1 = Conv2D(64, (3,3), activation = 'relu', padding='same')(inputs)
    conv1 = Conv2D(64, (3,3), activation = 'relu', padding='same')(conv1)
    pool1 = MaxPool2D((2,2))(conv1)

    conv2 = Conv2D(128, (3,3), activation = 'relu', padding='same')(pool1)
    conv2 = Conv2D(128, (3,3), activation = 'relu', padding='same')(conv2)
    pool2 = MaxPool2D((2,2))(conv2)

    conv3 = Conv2D(256, (3,3), activation = 'relu', padding='same')(pool2)
    conv3 = Conv2D(256, (3,3), activation = 'relu', padding='same')(conv3)
    pool3 = MaxPool2D((2,2))(conv3)

    conv4 = Conv2D(512, (3,3), activation = 'relu', padding='same')(pool3)
    conv4 = Conv2D(512, (3,3), activation = 'relu', padding='same')(conv4)

    goUp1 = Conv2DTranspose(256,(2,2),strides=(2,2),padding='same')(conv4)
    goUp1 = concatenate([goUp1,conv3])
    conv5 = Conv2D(256, (3,3), activation = 'relu', padding='same')(goUp1)
    conv5 = Conv2D(256, (3,3), activation = 'relu', padding='same')(conv5)

    goUp2 = Conv2DTranspose(128,(2,2),strides=(2,2),padding='same')(conv5)
    goUp2 = concatenate([goUp2,conv2])
    conv6 = Conv2D(128, (3,3), activation = 'relu', padding='same')(goUp2)
    conv6 = Conv2D(128, (3,3), activation = 'relu', padding='same')(conv6)

    goUp3 = Conv2DTranspose(64,(2,2),strides=(2,2),padding='same')(conv6)
    goUp3 = concatenate([goUp3,conv1])
    conv7 = Conv2D(64, (3,3), activation = 'relu', padding='same')(goUp3)
    conv7 = Conv2D(64, (3,3), activation = 'relu', padding='same')(conv7)



    outputs = Conv2D(1,(1,1),activation="sigmoid")(conv7)

    model = Model(inputs=[inputs],outputs=[outputs])
    return model

model = Unet()
model.compile(
              optimizer = 'adam', loss = 'binary_crossentropy',
              metrics = ['accuracy']
              )
     

In [6]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 128, 128,  │      1,792 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 128, 128,  │     36,928 │ conv2d_15[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 64, 64,    │          0 │ conv2d_16[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 64, 64,    │     73,856 │ max_pooling2d_3[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 64, 64,    │    147,584 │ conv2d_17[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 32, 32,    │          0 │ conv2d_18[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 32, 32,    │    295,168 │ max_pooling2d_4[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 32, 32,    │    590,080 │ conv2d_19[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 16, 16,    │          0 │ conv2d_20[0][0]   │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 16, 16,    │  1,180,160 │ max_pooling2d_5[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 16, 16,    │  2,359,808 │ conv2d_21[0][0]   │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_3  │ (None, 32, 32,    │    524,544 │ conv2d_22[0][0]   │
│ (Conv2DTranspose)   │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 32, 32,    │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 512)              │            │ conv2d_20[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 32, 32,    │  1,179,904 │ concatenate_3[0]… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 32, 32,    │    590,080 │ conv2d_23[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_4  │ (None, 64, 64,    │    131,200 │ conv2d_24[0][0] 

 Total params: 7,697,345 (29.36 MB)

 Trainable params: 7,697,345 (29.36 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
learning_rate_reduction = ReduceLROnPlateau(monitor = 'val_accuracy', patience = 2, verbose = 1, factor = 0.3, min_lr = 0.000001)

In [ ]:
batch_size = 32
n_epochs = 30
model.fit(x_train, y_train, batch_size = batch_size,
                    epochs = n_epochs, validation_data =(x_val, y_val),
                    callbacks = [learning_rate_reduction])

Epoch 1/30
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 186s 178ms/step - accuracy: 0.3256 - loss: 2.3297 - val_accuracy: 0.4232 - val_loss: 1.8937 - learning_rate: 0.0010
Epoch 2/30
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 171s 166ms/step - accuracy: 0.3181 - loss: 1.9369 - val_accuracy: 0.3034 - val_loss: 1.9332 - learning_rate: 0.0010
Epoch 3/30
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 0.2833 - loss: 1.8670
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 171s 167ms/step - accuracy: 0.2886 - loss: 1.8488 - val_accuracy: 0.4065 - val_loss: 1.8866 - learning_rate: 0.0010
Epoch 4/30
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 170s 165ms/step - accuracy: 0.2833 - loss: 1.7086 - val_accuracy: 0.3437 - val_loss: 1.5510 - learning_rate: 3.0000e-04
Epoch 5/30
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - accuracy: 0.2952 - loss: 1.6559
Epoch 5: ReduceLROnPlateau reducing learning rate to 9.000000427477062e-05.
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 168s 164ms/s

In [13]:
gc.collect()
print("Accuracy of the model is - " , model.evaluate(x_test,y_test)[1]*100 , "%")
model.save('my_model_full.keras')

129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 0.8545 - loss: 0.6097
Accuracy of the model is -  85.45365929603577 %


In [18]:
img = cv2.imread("music_dataset_spectro_3_instrument/train/Drum_set/222_spect.png")
img=cv2.resize(img,(img_size,img_size))
img= np.reshape(img,(-1, img_size, img_size, 3))
img=img/255

print(labels)
labels=np.array(labels)
pred=model.predict(img)
prediction = np.argmax(pred,1)
print(labels[prediction])


['Accordion' 'Acoustic_Guitar' 'Banjo' 'Bass_Guitar' 'Clarinet' 'cowbell'
 'Dobro' 'Drum_set' 'Electric_Guitar' 'flute' 'Harmonium' 'Horn'
 'Keyboard' 'Mandolin' 'Organ' 'Piano' 'Saxophone' 'Shakers' 'Tambourine'
 'Trombone' 'Trumpet' 'Ukulele' 'vibraphone' 'Violin']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
['Drum_set']
